# Experiment Analysis

基于 experiment_session_profile 评估历史 Billing Page 实验。该 Notebook 仅输出分析结果，不给出上线或业务建议。

## 1. Metric Framework

- Primary：Billing-to-Purchase Conversion Rate = 购买 Session / Billing 实验 Session。
- Diagnostic：Billing Abandonment Rate = 1 - Conversion Rate，仅用于解释流失。
- Guardrails：Revenue per Billing Session、Net Revenue per Billing Session、Refund Rate。
- Refund Rate = Refunded Orders / Purchased Orders。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

pd.set_option('display.float_format', lambda x: f'{x:,.6f}')
OUTPUT_DIR = Path('../output/tables')
profile = pd.read_csv(OUTPUT_DIR / 'experiment_session_profile.csv', parse_dates=['billing_exposure_at'])
assert profile['website_session_id'].is_unique
print(f'Experiment Sessions: {len(profile):,}')

Experiment Sessions: 1,311


## 2. Primary Metric

In [2]:
def conversion_result(data, sample_name):
    group = (data.groupby('billing_version')
             .agg(sessions=('website_session_id', 'size'), purchases=('purchased', 'sum'))
             .reindex(['control', 'treatment']))
    group['conversion_rate'] = group['purchases'] / group['sessions']
    p_control = group.loc['control', 'conversion_rate']
    p_treatment = group.loc['treatment', 'conversion_rate']
    n_control = int(group.loc['control', 'sessions'])
    n_treatment = int(group.loc['treatment', 'sessions'])
    absolute_lift = p_treatment - p_control
    z_stat, p_value = proportions_ztest(
        [int(group.loc['treatment', 'purchases']), int(group.loc['control', 'purchases'])],
        [n_treatment, n_control], alternative='two-sided')
    se = np.sqrt(p_control*(1-p_control)/n_control + p_treatment*(1-p_treatment)/n_treatment)
    return {
        'sample': sample_name, 'control_sessions': n_control, 'treatment_sessions': n_treatment,
        'control_purchases': int(group.loc['control', 'purchases']),
        'treatment_purchases': int(group.loc['treatment', 'purchases']),
        'control_conversion_rate': p_control, 'treatment_conversion_rate': p_treatment,
        'absolute_lift': absolute_lift, 'relative_lift': absolute_lift/p_control,
        'z_statistic': z_stat, 'p_value': p_value,
        'ci_lower': absolute_lift-1.96*se, 'ci_upper': absolute_lift+1.96*se}

primary = pd.DataFrame([conversion_result(profile, 'A_all_sessions')])
primary.to_csv(OUTPUT_DIR / 'experiment_primary_metric.csv', index=False)
primary

,sample,control_sessions,treatment_sessions,control_purchases,treatment_purchases,control_conversion_rate,treatment_conversion_rate,absolute_lift,relative_lift,z_statistic,p_value,ci_lower,ci_upper
0,A_all_sessions,657,654,300,410,0.456621,0.626911,0.170290,0.372936,6.187232,0.000000,0.117143,0.223438


In [3]:
primary_display = pd.DataFrame({
    'Metric': ['Sessions', 'Purchases', 'Conversion Rate'],
    'Control': [int(primary.control_sessions.iloc[0]), int(primary.control_purchases.iloc[0]), f'{primary.control_conversion_rate.iloc[0]:.2%}'],
    'Treatment': [int(primary.treatment_sessions.iloc[0]), int(primary.treatment_purchases.iloc[0]), f'{primary.treatment_conversion_rate.iloc[0]:.2%}']})
effect_display = pd.Series({
    'Absolute Lift': f'{primary.absolute_lift.iloc[0]*100:.2f} pp',
    'Relative Lift': f'{primary.relative_lift.iloc[0]:.2%}',
    'Z statistic': f'{primary.z_statistic.iloc[0]:.4f}',
    'p-value': f'{primary.p_value.iloc[0]:.3e}',
    '95% CI': f'[{primary.ci_lower.iloc[0]*100:.2f}, {primary.ci_upper.iloc[0]*100:.2f}] pp'})
display(primary_display)
display(effect_display.to_frame('Result'))

,Metric,Control,Treatment
0,Sessions,657,654
1,Purchases,300,410
2,Conversion Rate,45.66%,62.69%


,Result
Absolute Lift,17.03 pp
Relative Lift,37.29%
Z statistic,6.1872
p-value,6.123e-10
95% CI,"[11.71, 22.34] pp"


## 3. Diagnostic Metric

Billing Abandonment Rate 是主指标的补数，只用于解释 Billing 后流失，不作为独立显著性结论。

In [4]:
diagnostic = (profile.groupby('billing_version')
              .agg(sessions=('website_session_id', 'size'), purchases=('purchased', 'sum'))
              .reindex(['control', 'treatment']).reset_index())
diagnostic['billing_abandonment_rate'] = 1 - diagnostic['purchases']/diagnostic['sessions']
diagnostic.to_csv(OUTPUT_DIR / 'experiment_diagnostic_metrics.csv', index=False)
diagnostic

,billing_version,sessions,purchases,billing_abandonment_rate
0,control,657,300,0.543379
1,treatment,654,410,0.373089


## 4. Guardrail Metrics

Gross 与 Net Revenue 以全部 Billing Session 为分母；Refund Rate 以已购买订单为分母。收入均值差采用用户级 Bootstrap，保留同一用户多个 Session 的相关性。

In [5]:
guardrail = (profile.groupby('billing_version')
             .agg(sessions=('website_session_id','size'), orders=('purchased','sum'),
                  gross_revenue=('gross_revenue','sum'), net_revenue=('net_revenue','sum'),
                  refunded_orders=('refunded_order_flag','sum'))
             .reindex(['control','treatment']))
guardrail['revenue_per_billing_session'] = guardrail['gross_revenue']/guardrail['sessions']
guardrail['net_revenue_per_billing_session'] = guardrail['net_revenue']/guardrail['sessions']
guardrail['refund_rate'] = guardrail['refunded_orders']/guardrail['orders']
guardrail.reset_index().to_csv(OUTPUT_DIR / 'experiment_guardrail_metrics.csv', index=False)
guardrail

,sessions,orders,gross_revenue,net_revenue,refunded_orders,revenue_per_billing_session,net_revenue_per_billing_session,refund_rate
billing_version,,,,,,,,
control,657,300,"14,997.000000","13,997.200000",20,22.826484,21.304718,0.066667
treatment,654,410,"20,495.900000","18,546.290000",39,31.339297,28.358242,0.095122


### Refund Rate Guardrail Test

Refund Rate保持Guardrail定位，不属于Primary Metric。检验分母为各组已购买订单数量，比较Treatment与Control的退款订单比例。双侧两比例Z检验用于描述当前样本的不确定性；未达到显著性不能解释为两组相同或退款风险已被排除。


In [6]:
refund_successes = np.array([
    int(guardrail.loc['treatment', 'refunded_orders']),
    int(guardrail.loc['control', 'refunded_orders'])])
refund_orders = np.array([
    int(guardrail.loc['treatment', 'orders']),
    int(guardrail.loc['control', 'orders'])])
refund_z_stat, refund_p_value = proportions_ztest(
    refund_successes, refund_orders, alternative='two-sided')
refund_treatment = refund_successes[0] / refund_orders[0]
refund_control = refund_successes[1] / refund_orders[1]
refund_difference = refund_treatment - refund_control
refund_se = np.sqrt(
    refund_treatment * (1-refund_treatment) / refund_orders[0]
    + refund_control * (1-refund_control) / refund_orders[1])
refund_rate_test = pd.DataFrame([{
    'metric': 'Refund Rate (Guardrail)',
    'denominator': 'purchased orders',
    'control_refund_rate': refund_control,
    'treatment_refund_rate': refund_treatment,
    'difference_treatment_minus_control': refund_difference,
    'z_statistic': refund_z_stat,
    'p_value': refund_p_value,
    'ci_lower': refund_difference - 1.96*refund_se,
    'ci_upper': refund_difference + 1.96*refund_se}])
refund_rate_test


,metric,denominator,control_refund_rate,treatment_refund_rate,difference_treatment_minus_control,z_statistic,p_value,ci_lower,ci_upper
0,Refund Rate (Guardrail),purchased orders,0.066667,0.095122,0.028455,1.356839,0.174833,-0.011586,0.068496


In [7]:
users = np.array(sorted(profile['user_id'].unique()))
user_group = (profile.groupby(['user_id','billing_version'])
              .agg(sessions=('website_session_id','size'), gross=('gross_revenue','sum'), net=('net_revenue','sum'))
              .unstack(fill_value=0).reindex(users, fill_value=0))
rng = np.random.default_rng(20260821)
repetitions = 10_000
bootstrap_differences = np.empty((repetitions, 2))
for i in range(repetitions):
    weights = rng.multinomial(len(users), np.full(len(users), 1/len(users)))
    for j, metric in enumerate(['gross','net']):
        control_mean = (weights*user_group[(metric,'control')].to_numpy()).sum() / (weights*user_group[('sessions','control')].to_numpy()).sum()
        treatment_mean = (weights*user_group[(metric,'treatment')].to_numpy()).sum() / (weights*user_group[('sessions','treatment')].to_numpy()).sum()
        bootstrap_differences[i,j] = treatment_mean-control_mean

bootstrap = pd.DataFrame([
 {'metric':'Revenue per Billing Session',
  'control_value':guardrail.loc['control','revenue_per_billing_session'],
  'treatment_value':guardrail.loc['treatment','revenue_per_billing_session'],
  'difference':guardrail.loc['treatment','revenue_per_billing_session']-guardrail.loc['control','revenue_per_billing_session'],
  'bootstrap_ci_lower':np.quantile(bootstrap_differences[:,0],.025),
  'bootstrap_ci_upper':np.quantile(bootstrap_differences[:,0],.975)},
 {'metric':'Net Revenue per Billing Session',
  'control_value':guardrail.loc['control','net_revenue_per_billing_session'],
  'treatment_value':guardrail.loc['treatment','net_revenue_per_billing_session'],
  'difference':guardrail.loc['treatment','net_revenue_per_billing_session']-guardrail.loc['control','net_revenue_per_billing_session'],
  'bootstrap_ci_lower':np.quantile(bootstrap_differences[:,1],.025),
  'bootstrap_ci_upper':np.quantile(bootstrap_differences[:,1],.975)}])
bootstrap['bootstrap_unit']='user'
bootstrap['bootstrap_repetitions']=repetitions
bootstrap['random_seed']=20260821
bootstrap.to_csv(OUTPUT_DIR / 'experiment_bootstrap_results.csv', index=False)
bootstrap

,metric,control_value,treatment_value,difference,bootstrap_ci_lower,bootstrap_ci_upper,bootstrap_unit,bootstrap_repetitions,random_seed
0,Revenue per Billing Session,22.826484,31.339297,8.512813,5.846222,11.134826,user,10000,20260821
1,Net Revenue per Billing Session,21.304718,28.358242,7.053523,4.366346,9.691680,user,10000,20260821


## 5. User Cross-group Sensitivity Analysis

In [8]:
analysis_samples = {
 'A_all_sessions': profile,
 'B_exclude_cross_group_users': profile[profile['user_cross_group_flag']==0].copy(),
 'C_first_experiment_session_per_user': (profile.sort_values(['user_id','billing_exposure_at','billing_pageview_id'])
                                         .drop_duplicates('user_id'))}
sensitivity = pd.DataFrame([conversion_result(data,name) for name,data in analysis_samples.items()])
sensitivity.to_csv(OUTPUT_DIR / 'experiment_sensitivity_analysis.csv', index=False)
sensitivity_display = sensitivity[['sample','control_sessions','treatment_sessions',
 'control_conversion_rate','treatment_conversion_rate','absolute_lift','relative_lift','p_value']].copy()
for col in ['control_conversion_rate','treatment_conversion_rate','relative_lift']:
    sensitivity_display[col] = sensitivity_display[col].map(lambda x: f'{x:.2%}')
sensitivity_display['absolute_lift'] = sensitivity_display['absolute_lift'].map(lambda x: f'{x*100:.2f} pp')
sensitivity_display['p_value'] = sensitivity_display['p_value'].map(lambda x: f'{x:.3e}')
sensitivity_display

,sample,control_sessions,treatment_sessions,control_conversion_rate,treatment_conversion_rate,absolute_lift,relative_lift,p_value
0,A_all_sessions,657,654,45.66%,62.69%,17.03 pp,37.29%,6.123e-10
1,B_exclude_cross_group_users,651,648,45.47%,62.65%,17.19 pp,37.80%,5.153e-10
2,C_first_experiment_session_per_user,650,651,45.54%,62.67%,17.13 pp,37.63%,5.601e-10


## 6. Analysis Output Scope

本阶段仅形成主指标、诊断指标、护栏指标、Bootstrap 区间和敏感性分析结果表。业务含义与是否扩大流量将在下一阶段单独讨论。